In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
import plotly.express as px
import numpy as np

In [287]:
df = pd.read_csv("movies.csv")
df = df.dropna(subset=["overview"])
df

,names,date_x,score,genre,overview,crew,orig_title,status,orig_lang,budget_x,revenue,country
0,Creed III,03/02/2023,73.0,"Drama, Action","After dominating the boxing world, Adonis Cree...","Michael B. Jordan, Adonis Creed, Tessa Thompso...",Creed III,Released,English,75000000.0,2.716167e+08,AU
1,Avatar: The Way of Water,12/15/2022,78.0,"Science Fiction, Adventure, Action",Set more than a decade after the events of the...,"Sam Worthington, Jake Sully, Zoe Saldaña, Neyt...",Avatar: The Way of Water,Released,English,460000000.0,2.316795e+09,AU
2,The Super Mario Bros. Movie,04/05/2023,76.0,"Animation, Adventure, Family, Fantasy, Comedy","While working underground to fix a water main,...","Chris Pratt, Mario (voice), Anya Taylor-Joy, P...",The Super Mario Bros. Movie,Released,English,100000000.0,7.244590e+08,AU
3,Mummies,01/05/2023,70.0,"Animation, Comedy, Family, Adventure, Fantasy","Through a series of unfortunate events, three ...","Óscar Barberán, Thut (voice), Ana Esther Albor...",Momias,Released,"Spanish, Castilian",12300000.0,3.420000e+07,AU
4,Supercell,03/17/2023,61.0,Action,Good-hearted teenager William always lived in ...,"Skeet Ulrich, Roy Cameron, Anne Heche, Dr Quin...",Supercell,Released,English,77000000.0,3.409420e+08,US
...,...,...,...,...,...,...,...,...,...,...,...,...
10173,20th Century Women,12/28/2016,73.0,Drama,"In 1979 Santa Barbara, California, Dorothea Fi...","Annette Bening, Dorothea Fields, Lucas Jade Zu...",20th Century Women,Released,English,7000000.0,9.353729e+06,US
10174,Delta Force 2: The Colombian Connection,08/24/1990,54.0,Action,When DEA agents are taken captive by a ruthles...,"Chuck Norris, Col. Scott McCoy, Billy Drago, R...",Delta Force 2: The Colombian Connection,Released,English,9145817.8,6.698361e+06,US
10175,The Russia House,12/21/1990,61.0,"Drama, Thriller, Romance","Barley Scott Blair, a Lisbon-based editor of R...","Sean Connery, Bartholomew 'Barley' Scott Blair...",The Russia House,Released,English,21800000.0,2.299799e+07,US
10176,Darkman II: The Return of Durant,07/11/1995,55.0,"Action, Adventure, Science Fiction, Thriller, ...",Darkman and Durant return and they hate each o...,"Larry Drake, Robert G. Durant, Arnold Vosloo, ...",Darkman II: The Return of Durant,Released,English,116000000.0,4.756613e+08,US


In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["overview"].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\x1Ras\Desktop\Coding\ML-Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\x1Ras\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/319 [00:00<?, ?it/s]

In [286]:
reducer = umap.UMAP(
    n_components=3,
    random_state=42
)
coords = reducer.fit_transform(embeddings)

c:\Users\x1Ras\Desktop\Coding\ML-Project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [288]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=2)
clusters = clusterer.fit_predict(coords)

In [289]:
coords.shape

(10178, 3)

In [290]:
mean = coords.mean(axis=0)
std = coords.std(axis=0)

z_scores = np.abs((coords - mean) / std)

mask = np.all(z_scores < 4, axis=1)

coords = coords[mask]
clusters = clusters[mask]
df = df[mask]
coords

array([[10.779449 ,  3.0979798,  2.5165033],
       [11.804087 ,  3.9509883,  4.1878586],
       [10.520637 ,  2.4335282,  4.074416 ],
       ...,
       [13.476842 ,  3.1603749,  2.3634434],
       [10.634898 ,  3.0716012,  2.4433587],
       [10.017227 ,  3.4381065,  5.2315207]],
      shape=(10089, 3), dtype=float32)

In [291]:
fig = px.scatter_3d(
    x=coords[:,0],
    y=coords[:,1],
    z=coords[:,2],
    color=clusters.astype(str),
    hover_name=df["names"]
)

fig.update_layout(
    width=1200,
    height=800,
    scene=dict(
        aspectmode='data'
    )
)

fig.show()

In [283]:
mask = clusters != -1

clusters_clean = clusters[mask]
coords_clean = coords[mask]
df_clean = df[mask]

In [292]:
coords_clean.shape

(6570, 3)

In [293]:
fig1 = px.scatter_3d(
    x=coords_clean[:,0],
    y=coords_clean[:,1],
    z=coords_clean[:,2],
    color=clusters_clean.astype(str),
    hover_name=df_clean["names"]
)

fig1.update_layout(
    width=1200,
    height=800,
    margin=dict(l=0, r=0, t=0, b=0),
    scene=dict(
        aspectmode='data'
    )
)

fig1.show()